# Train Model


ในที่นี้จะเป็นการพวม Step ในการ Train Model

## Load Library


In [ ]:
!pip install pythainlp
!pip install gensim
!pip install deepcut


  Using cached deepcut-0.7.0.0-py3-none-any.whl.metadata (600 bytes)
Using cached deepcut-0.7.0.0-py3-none-any.whl (2.0 MB)


In [ ]:
import tensorflow as tf
import numpy as np
import re
import io
import random
import math
import os
from sklearn.model_selection import train_test_split
from collections import Counter, defaultdict
from pythainlp.tokenize import word_tokenize
from keras.models import load_model
from tensorflow.keras import layers, models





## Connect Storege

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CLASS_NAMES = ['E', 'P', 'T']
label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
inverse_label_map = {i: name for i, name in enumerate(CLASS_NAMES)}

#ใส่ File ต้นแบบก่อน
#Google Coulds
input_file_path = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_input.txt'
ans_file_path = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_ans.txt'

In [ ]:
def load_data(sentences_path, labels_path):
    """Loads sentences and labels from the provided files."""
    sentences_dict = {}
    labels_dict = {}

    # Read sentences
    with open(sentences_path, 'r', encoding='utf-8') as f:
        for line in f:
            if '::' in line:
                parts = line.strip().split('::', 1)
                try:
                    line_num = int(parts[0])
                    # print(parts[0]) tag
                    # print(parts[1]) sentan
                    sentences_dict[line_num] = parts[1]
                except ValueError:
                    continue # Skip lines with non-integer keys

    # Read labels
    with open(labels_path, 'r', encoding='utf-8') as f:
        for line in f:
            if '::' in line:
                parts = line.strip().split('::', 1)
                try:
                    line_num = int(parts[0])
                    # Labels can be comma-separated, so we split them
                    labels_dict[line_num] = [label.strip() for label in parts[1].split(',')]
                except ValueError:
                    continue # Skip lines with non-integer keys

    return sentences_dict, labels_dict

## Tokenize

In [ ]:
def tokenize_sentences(sentences):
  """Tokenizes the sentences using pythainlp."""
  print(sentences)
  tokenized_sentences = {}
  for line_num, sentence in sentences.items():
    tokenized_sentences[line_num] = word_tokenize(sentence , engine = 'deepcut' )
    # tokenized_sentences[line_num] = word_tokenize(sentence  )

  return tokenized_sentences

In [ ]:
sentences, labels = load_data(input_file_path, ans_file_path)

print(sentences)
print()
print(labels)
# Print a few examples to verify
print("--- Example Data ---")
for i in range(1, 50):
     if i in sentences and i in labels:
         print(f"Sentence {i}: {sentences[i]}")
         print(f"Labels {i}: {labels[i]}\n")

{1: 'พรุ่งนี้ตาจะมาถึงเชียงใหม่ช่วงบ่ายโมงวันนี้', 2: 'คุณตาเรียกให้ฉันไปช่วยกวาดบ้านตอนเช้า', 3: 'ขณะขับรถ เราไม่ควรให้น้องหมายื่นหัวออกจากหน้าต่าง เพราะอาจทำให้น้องหมาตาแห้งได้', 4: 'ถ้าอยากเห็นผีเราต้องเปิดดวงตาที่สามให้ได้', 5: 'ฮู้ แพ้อีกแล้วยังไม่ชนะเลยเล่นอีกตาไหม', 6: 'ตาของผมแดงเพราะเล่นเกมเอลโอเอลไปสิบตาติดทั้งคืน', 7: 'ตาท้ายไม่มีอยู่จริงเพราะว่าเรายังแพ้อยู่งั้นเล่นอีกตา', 8: 'คุณตาตะกูนอูจิวะมีเนตรวงแหวนซึ่งทำให้ดวงตาของเขาเปลี่ยนไป', 9: 'คุณตากล่าวว่า ความหวาดกลัวมนุษย์ไม่อาจมองเห็นด้วยตาเปล่า แต่สามารถรับรู้ได้ผ่านสัมผัสอื่น', 10: 'ตาซ้ายกับตาขวาของผมใหญ่ไม่เท่ากัน', 11: 'คุณตาเล่นหมากรุกเก่งมาก ผมว่า คงต้องนัดล้างตาอีกสักสองสามตาแล้วล่ะครับ', 12: 'แพ้มากี่ตาแล้วเนี่ย คุณตานั้นเก่งชะมัด เราควรจับตามองไว้', 13: 'ตาแล้วตาเล่าผมก็ยังเล่นหมากรุกแพ้คุณตา', 14: 'คุณตาบอกว่าตอนเด็กๆ เคยเจอตาทวดที่ตายไปแล้ว ยืนมองเขาด้วยสายตาอันเย็นชา', 15: 'คุณหมอตรวจดวงตาของคุณตาเพื่อเช็กสุขภาพดวงตา', 16: 'ตาสีตาสามาเดินด้วยกัน', 17: 'อดนอนจนตาโบ๋ไปหมดแล้ว', 18: 'ปอปลาตากลมผอผึ้งทำรัง', 19: 'ห

In [ ]:

length_sen = len(sentences)

first_10_sentences = {key: sentences[key] for key in range(0, length_sen) if key in sentences}

# แสดงผลลัพธ์
print(first_10_sentences)

tokenized_sentences = tokenize_sentences(first_10_sentences)
print("\n--- Tokenized Sentences ---")
for i in range(1, min(len(tokenized_sentences) + 1, length_sen)):
  print(f"Sentence {i}: {tokenized_sentences[i]}")


{1: 'พรุ่งนี้ตาจะมาถึงเชียงใหม่ช่วงบ่ายโมงวันนี้', 2: 'คุณตาเรียกให้ฉันไปช่วยกวาดบ้านตอนเช้า', 3: 'ขณะขับรถ เราไม่ควรให้น้องหมายื่นหัวออกจากหน้าต่าง เพราะอาจทำให้น้องหมาตาแห้งได้', 4: 'ถ้าอยากเห็นผีเราต้องเปิดดวงตาที่สามให้ได้', 5: 'ฮู้ แพ้อีกแล้วยังไม่ชนะเลยเล่นอีกตาไหม', 6: 'ตาของผมแดงเพราะเล่นเกมเอลโอเอลไปสิบตาติดทั้งคืน', 7: 'ตาท้ายไม่มีอยู่จริงเพราะว่าเรายังแพ้อยู่งั้นเล่นอีกตา', 8: 'คุณตาตะกูนอูจิวะมีเนตรวงแหวนซึ่งทำให้ดวงตาของเขาเปลี่ยนไป', 9: 'คุณตากล่าวว่า ความหวาดกลัวมนุษย์ไม่อาจมองเห็นด้วยตาเปล่า แต่สามารถรับรู้ได้ผ่านสัมผัสอื่น', 10: 'ตาซ้ายกับตาขวาของผมใหญ่ไม่เท่ากัน', 11: 'คุณตาเล่นหมากรุกเก่งมาก ผมว่า คงต้องนัดล้างตาอีกสักสองสามตาแล้วล่ะครับ', 12: 'แพ้มากี่ตาแล้วเนี่ย คุณตานั้นเก่งชะมัด เราควรจับตามองไว้', 13: 'ตาแล้วตาเล่าผมก็ยังเล่นหมากรุกแพ้คุณตา', 14: 'คุณตาบอกว่าตอนเด็กๆ เคยเจอตาทวดที่ตายไปแล้ว ยืนมองเขาด้วยสายตาอันเย็นชา', 15: 'คุณหมอตรวจดวงตาของคุณตาเพื่อเช็กสุขภาพดวงตา', 16: 'ตาสีตาสามาเดินด้วยกัน', 17: 'อดนอนจนตาโบ๋ไปหมดแล้ว', 18: 'ปอปลาตากลมผอผึ้งทำรัง', 19: 'ห

KeyboardInterrupt: 

## Suffle Data

In [ ]:
import random

# สมมติว่าคุณมี X_INPUT และ Y_OUTPUT
X_INPUT = tokenized_sentences  # dict เช่น {1: [...], 2: [...], ...}
Y_OUTPUT = {key: labels[key] for key in range(0, length_sen) if key in labels}

# 1. สร้างรายการของคู่ (id, X, Y)
pairs = [(key, X_INPUT[key], Y_OUTPUT[key]) for key in Y_OUTPUT.keys() if key in X_INPUT]
print(pairs)
# 2. สุ่มลำดับ
random.seed(42)
random.shuffle(pairs)

# 3. แยกกลับออกเป็น dict ใหม่
X_SHUFFLED = {i+1: pair[1] for i, pair in enumerate(pairs)}
Y_SHUFFLED = {i+1: pair[2] for i, pair in enumerate(pairs)}

# ✅ ตรวจสอบตัวอย่างหลัง shuffle


print(X_SHUFFLED)
print(Y_SHUFFLED)


NameError: name 'tokenized_sentences' is not defined

ทำ

In [ ]:

# ใช้ค่า map เดิมจากเซลล์ของคุณ
# CLASS_NAMES, label_map, inverse_label_map
# X_INPUT = {line_id: [tokens...]}  (tokenized)
# Y_OUTPUT = {line_id: [labels for each 'ตา' occurrence in this sentence]}
TARGET_PAT = r'ตา'  # เกณฑ์ง่าย: มี 'ตา' อยู่ที่ไหนในโทเค็นนั้นบ้าง (รวมคำประสม เช่น สายตา, แว่นตา, เบ้าตา ฯลฯ)
WINDOW = 6          # จำนวนโทเค็นซ้าย/ขวาเป็นบริบท
SENTINEL = '[TARGET]'

def find_ta_indices(tokens):
    """คืน index ของทุก token ที่มี 'ตา' เป็นส่วนหนึ่ง."""
    idxs = []
    for i, tok in enumerate(tokens):
        if 'ตา' in tok:  # เกณฑ์ง่าย ตรงกับรูปข้อมูลที่ให้มา
            idxs.append(i)
    return idxs

def build_samples(x_input, y_output):
    texts, labels = [], []
    dropped_mismatch = 0
    per_sent_stats = []

    for line_id, tokens in x_input.items():
        target_positions = find_ta_indices(tokens)
        gold = y_output.get(line_id, [])

        # จัดการจำนวน label ไม่เท่ากับจำนวนตำแหน่ง 'ตา'
        m = min(len(target_positions), len(gold))
        if len(target_positions) != len(gold):
            dropped_mismatch += 1
        # เอาเฉพาะส่วนที่ match ได้
        for k in range(m):
            pos = target_positions[k]
            left = max(0, pos - WINDOW)
            right = min(len(tokens), pos + WINDOW + 1)
            context = tokens[left:pos] + [SENTINEL] + tokens[pos+1:right]
            text = ' '.join(context)  # จะให้ TextVectorization split ด้วย whitespace
            lab = gold[k]
            if lab not in label_map:
                # ข้าม label แปลก
                continue
            texts.append(text)
            labels.append(label_map[lab])

        per_sent_stats.append((line_id, len(target_positions), len(gold), m))

    print(f"จำนวนประโยคทั้งหมด: {len(x_input)}")
    print(f"จำนวนตัวอย่าง (token-level) ที่สร้างได้: {len(texts)}")
    if dropped_mismatch:
        print(f"คำเตือน: มี {dropped_mismatch} ประโยคที่จำนวนตำแหน่ง 'ตา' ไม่เท่ากับจำนวน label (ตัดให้เท่ากันโดยอัตโนมัติ)")

    return texts, np.array(labels, dtype=np.int32)

In [ ]:
# ===== Cell: Build supervised samples per target token =====
train_texts, train_labels = build_samples(X_SHUFFLED, Y_SHUFFLED)
print(train_texts , train_labels)
# สรุปคลาส
cnt = Counter(train_labels.tolist())
print("Class distribution (by index):", cnt, " => ", {inverse_label_map[i]: c for i, c in cnt.items()})


จำนวนประโยคทั้งหมด: 99
จำนวนตัวอย่าง (token-level) ที่สร้างได้: 168
คำเตือน: มี 14 ประโยคที่จำนวนตำแหน่ง 'ตา' ไม่เท่ากับจำนวน label (ตัดให้เท่ากันโดยอัตโนมัติ)
['รอบ นี้ [TARGET] ของ ฉัน ที่ ต้อง พูด บาง', 'รู้ ทุก ครั้ง ที่ เรา มอง [TARGET]   ยัง ทำ ให้ ใจ ของ', 'ถึง [TARGET] จะ แก่ มาก แล้ว แต่ ตาตา', 'ตา จะ แก่ มาก แล้ว แต่ [TARGET] ยัง ดี อยู่ เลย', '[TARGET] ซ้าย กับ ตาขวา ของ ผม ใหญ่', 'ตา ซ้าย กับ [TARGET] ของ ผม ใหญ่ ไม่ เท่า กัน', '[TARGET] เปล่ง แสง เป็น ประกาย   หลัง', 'หลัง จาก เริ่ม จั่ว การ์ด ของ [TARGET] นี้', '[TARGET] ถัด ไป เรา จะ เล่น หมาก', '[TARGET] เรียก ให้ ฉัน ไป ช่วย กวาด', 'ฉัน เดิน ไป เจอ ผี [TARGET] เลย', '[TARGET] เดิน ด้วย กัน', 'แก ลอง มอง ใน [TARGET] เขา   ถ้า มี สายตา แบบ', 'ใน ตา เขา   ถ้า มี [TARGET] แบบ เดียว กับ ฉัน ที่ มอง', '[TARGET] นั่ง ตากลม ทำ ตา กลม', 'ตา นั่ง [TARGET] ทำ ตา กลม', 'คุณ [TARGET] เล่น หมาก รุก เก่ง มาก  ', 'ว่า   คง ต้อง นัด ล้าง [TARGET] อีก สัก สองสาม ตา แล้ว ล่ะ', 'นัด ล้าง ตา อีก สัก สองสาม [TARGET] แล้ว ล่ะ ครับ', '[TARGET

## ทำ word Vectorization ของ word

In [ ]:
def make_ds(texts, labels, batch_size=64, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts), seed=42)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ===== Cell: Split and vectorize =====


# สุ่มคงที่เพื่อ reproducible
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

X_train, X_val, y_train, y_val = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels if len(set(train_labels))>1 else None
)

vectorizer = tf.keras.layers.TextVectorization(
    standardize=None,              # ไม่แตะต้องสตริง (เราตัดคำแล้ว ใช้ whitespace แบ่ง)
    split='whitespace',
    max_tokens=30000,
    output_mode='int',
    output_sequence_length=50      # ตัด/แพดลำดับความยาว 50 โทเค็น (พอสำหรับ WINDOW สองข้าง)
)
vectorizer.adapt(X_train)

def make_ds(texts, labels, batch_size=64, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts), seed=42)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, batch_size=64, shuffle=True)
val_ds   = make_ds(X_val,   y_val,   batch_size=64, shuffle=False)
print(train_ds)
print(val_ds)


<_PrefetchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.string, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.string, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


## สร้าง Model

Load Model ถ้ามี

In [ ]:
model =  load_model('/content/drive/MyDrive/Colab_Notebooks/NLP_Model/ALL_Model/Model_1_V2.keras')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 13 variables whereas the saved optimizer has 24 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# ===== Rebuild model with scalar string input =====

num_classes = len(CLASS_NAMES)

inputs = layers.Input(shape=(), dtype=tf.string, name="text")   # << แก้ตรงนี้ จาก (None,) เป็น ()
x = vectorizer(inputs)
x = layers.Embedding(input_dim=len(vectorizer.get_vocabulary()), output_dim=128, mask_zero=True)(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)



In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 50)        │          0 │ text[0][0]        │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 50, 128)   │    503,424 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_7         │ (None, 50)        │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │     98,816 │ embedding_3[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 64)        │      8,256 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 3)         │        195 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 610,691 (2.33 MB)

 Trainable params: 610,691 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

## Train Model

In [ ]:
history = model.fit(
    train_ds,                         # ใช้ ds ตรง ๆ ได้เลย
    validation_data=val_ds,
    epochs=10,
    verbose=1
)

Epoch 1/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 29s 166ms/step - accuracy: 0.5498 - loss: 0.9086 - val_accuracy: 0.7469 - val_loss: 0.5981
Epoch 2/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 40s 160ms/step - accuracy: 0.8241 - loss: 0.4577 - val_accuracy: 0.7778 - val_loss: 0.5652
Epoch 3/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 22s 161ms/step - accuracy: 0.8739 - loss: 0.3408 - val_accuracy: 0.7953 - val_loss: 0.5546
Epoch 4/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 23s 165ms/step - accuracy: 0.9122 - loss: 0.2373 - val_accuracy: 0.7994 - val_loss: 0.5592
Epoch 5/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 21s 154ms/step - accuracy: 0.9414 - loss: 0.1717 - val_accuracy: 0.8107 - val_loss: 0.5889
Epoch 6/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 41s 154ms/step - accuracy: 0.9511 - loss: 0.1363 - val_accuracy: 0.8210 - val_loss: 0.6044
Epoch 7/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 26s 192ms/step - accuracy: 0.9641 - loss: 0.1108 - val_accuracy: 0.8138 - val_loss: 0.7160
Epoch 8/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 22s 164ms/step - accuracy: 0.9698 - loss: 0

## ทดสอบ Model

In [ ]:
def predict_labels_for_sentence(tokens):
    """คืน list ของ label ตัวอักษร (E/P/T) ตามจำนวน 'ตา' ที่เจอใน tokens"""
    positions = find_ta_indices(tokens)
    preds = []
    for pos in positions:
        left = max(0, pos - WINDOW)
        right = min(len(tokens), pos + WINDOW + 1)
        context = tokens[left:pos] + [SENTINEL] + tokens[pos+1:right]
        text = ' '.join(context)

        # ⬇️ แก้บรรทัดนี้
        # prob = model.predict([text], verbose=0)[0]
        prob = model.predict(np.array([text], dtype=object), verbose=0)[0]
        # หรือใช้อีกแบบ:
        # prob = model(np.array([text], dtype=object)).numpy()[0]

        pred_idx = int(np.argmax(prob))
        preds.append(inverse_label_map[pred_idx])
    return preds



def write_predictions(input_file, output_file):
    """อ่านประโยคจาก input_file (รูปแบบ id::ประโยค) แล้วเขียนผลพยากรณ์ลง output_file"""
    # 1) อ่านเป็น dict: {id(int): sentence(str)}
    sentences_only = {}
    with open(input_file, 'r', encoding='utf-8') as fin:
        for line in fin:
            line = line.strip()
            if not line or "::" not in line:
                continue
            line_id, text = line.split("::", 1)
            sentences_only[int(line_id)] = text.strip()

    # 2) ส่ง 'dict' เข้า tokenizer (ฟังก์ชันของคุณต้องการ .items())
    #    ผลลัพธ์คาดว่าเป็น {id: [tokens...]}
    tokenized_all = tokenize_sentences(sentences_only)
    print("Tokenized sentences:", tokenized_all)

    # 3) ทำนายและเขียนผลเป็น id::label[,label...]
    with open(output_file, 'w', encoding='utf-8') as fout:
        for line_id in sorted(tokenized_all.keys()):
            tokens = tokenized_all[line_id]
            preds = predict_labels_for_sentence(tokens)
            out = f"{line_id}::" + (",".join(preds) if preds else "")
            fout.write(out + "\n")

    print(f"✅ เขียนผลลัพธ์แล้วที่: {os.path.abspath(output_file)}")


In [ ]:
# ==== รัน ====
# 1::คุณดวงตาสวยมาก
# 2::คุณตาดูอะไรครับ
# 3::ตานี้ตาสุดท้าย
tmp = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_input.txt'   # ไฟล์ input เช่น 1::คุณดวงตาสวยมาก
tmpans = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_pred.txt'      # ผลลัพธ์ต้องการให้ชื่อ pred.txt ตามที่ระบุ

write_predictions(tmp, tmpans)

# แสดงผล 20 บรรทัดแรกของ pred.txt
print("\n=== ตัวอย่างผลลัพธ์จาก pred.txt ===")
with open(tmpans, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(line.strip())


{1: 'พรุ่งนี้ตาจะมาถึงเชียงใหม่ช่วงบ่ายโมงวันนี้', 2: 'คุณตาเรียกให้ฉันไปช่วยกวาดบ้านตอนเช้า', 3: 'ขณะขับรถ เราไม่ควรให้น้องหมายื่นหัวออกจากหน้าต่าง เพราะอาจทำให้น้องหมาตาแห้งได้', 4: 'ถ้าอยากเห็นผีเราต้องเปิดดวงตาที่สามให้ได้', 5: 'ฮู้ แพ้อีกแล้วยังไม่ชนะเลยเล่นอีกตาไหม', 6: 'ตาของผมแดงเพราะเล่นเกมเอลโอเอลไปสิบตาติดทั้งคืน', 7: 'ตาท้ายไม่มีอยู่จริงเพราะว่าเรายังแพ้อยู่งั้นเล่นอีกตา', 8: 'คุณตาตะกูนอูจิวะมีเนตรวงแหวนซึ่งทำให้ดวงตาของเขาเปลี่ยนไป', 9: 'คุณตากล่าวว่า ความหวาดกลัวมนุษย์ไม่อาจมองเห็นด้วยตาเปล่า แต่สามารถรับรู้ได้ผ่านสัมผัสอื่น', 10: 'ตาซ้ายกับตาขวาของผมใหญ่ไม่เท่ากัน', 11: 'คุณตาเล่นหมากรุกเก่งมาก ผมว่า คงต้องนัดล้างตาอีกสักสองสามตาแล้วล่ะครับ', 12: 'แพ้มากี่ตาแล้วเนี่ย คุณตานั้นเก่งชะมัด เราควรจับตามองไว้', 13: 'ตาแล้วตาเล่าผมก็ยังเล่นหมากรุกแพ้คุณตา', 14: 'คุณตาบอกว่าตอนเด็กๆ เคยเจอตาทวดที่ตายไปแล้ว ยืนมองเขาด้วยสายตาอันเย็นชา', 15: 'คุณหมอตรวจดวงตาของคุณตาเพื่อเช็กสุขภาพดวงตา', 16: 'ตาสีตาสามาเดินด้วยกัน', 17: 'อดนอนจนตาโบ๋ไปหมดแล้ว', 18: 'ปอปลาตากลมผอผึ้งทำรัง', 19: 'ห

In [ ]:
import os

# ---------- PATHS ----------
ANS_PATH   = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_ans.txt"
PRED_PATH  = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_pred.txt"
METRIC_A_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/metric_A2.txt"
METRIC_B_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/metric_B2.txt"

VALID_LABELS = {"E", "P", "T"}

# ---------- IO HELPERS ----------
def parse_labels_file(path):
    """
    อ่านไฟล์รูปแบบ:
        1::E,E
        2::P
        3::T,T
    แล้วคืน dict: {int_id: [label, ...]}
    """
    data = {}
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or "::" not in line:
                continue
            k, v = line.split("::", 1)
            k = int(k.strip())
            v = v.strip()
            if not v:
                labels = []
            else:
                labels = [lab.strip() for lab in v.split(",") if lab.strip()]
            data[k] = labels
    return data

# ---------- METRIC A ----------
def evaluate_metric_A(gold_dict, pred_dict, out_path):
    """
    First-Word Accuracy:
      - ต่อบรรทัด: "id:: Correct" ถ้า label ตัวแรกตรงกันและมีอยู่ทั้งคู่ มิฉะนั้น "Wrong"
      - สุดท้าย: "Accuracy::X/Y"
    เกณฑ์:
      - ถ้าฝั่งใดฝั่งหนึ่งไม่มี label ตัวแรก (ลิสต์ว่าง) => Wrong
    """
    ids = sorted(set(gold_dict.keys()) | set(pred_dict.keys()))
    correct = 0
    total = len(ids)

    lines = []
    for i in ids:
        gold = gold_dict.get(i, [])
        pred = pred_dict.get(i, [])
        if gold and pred and gold[0] == pred[0]:
            lines.append(f"{i}:: Correct")
            correct += 1
        else:
            lines.append(f"{i}:: Wrong")

    lines.append(f"Accuracy::{correct}/{total}")

    # write file
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return correct, total

# ---------- Levenshtein (token-level) ----------
def levenshtein_ops(seq_a, seq_b):
    """
    Levenshtein distance สำหรับลิสต์ของสัญลักษณ์ (เช่น ['E','E'] กับ ['P','E'])
    คืนจำนวน operations (insert/delete/replace) ขั้นต่ำเพื่อแปลง seq_a -> seq_b
    """
    na, nb = len(seq_a), len(seq_b)
    dp = [[0]*(nb+1) for _ in range(na+1)]
    for i in range(na+1):
        dp[i][0] = i
    for j in range(nb+1):
        dp[0][j] = j
    for i in range(1, na+1):
        for j in range(1, nb+1):
            cost = 0 if seq_a[i-1] == seq_b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # delete
                dp[i][j-1] + 1,      # insert
                dp[i-1][j-1] + cost  # replace / match
            )
    return dp[na][nb]

# ---------- METRIC B ----------
def evaluate_metric_B(gold_dict, pred_dict, out_path):
    """
    Metric B: Edit distance (token-level) ต่อบรรทัด + ค่าเฉลี่ย normalized
      - รายบรรทัด: "id:: <ops>/<len_pred>"
          * ops = Levenshtein distance(GOLD, PRED)
          * len_pred = จำนวน token ใน pred บรรทัดนั้น (รายงานตามจริง; ถ้า 0 จะพิมพ์ 0/0)
      - Avg::X.YZ (เฉลี่ย ops / max(1, len_pred))
        * ใช้ max(1, len_pred) เพื่อเลี่ยงหารศูนย์ แต่ในไฟล์บรรทัดยังคงโชว์ตัวหารจริง
    """
    ids = sorted(set(gold_dict.keys()) | set(pred_dict.keys()))
    norm_sum = 0.0
    count = 0

    lines = []
    for i in ids:
        gold = gold_dict.get(i, [])
        pred = pred_dict.get(i, [])
        ops = levenshtein_ops(gold, pred)
        denom_real = len(pred)               # แสดงผลตามที่ผู้ใช้ต้องการ
        denom_for_avg = max(1, denom_real)   # ป้องกันหารศูนย์ตอนเฉลี่ย

        lines.append(f"{i}:: {ops}/{denom_real}")
        norm_sum += ops / denom_for_avg
        count += 1

    avg = (norm_sum / count) if count else 0.0
    lines.append(f"Avg::{avg:.2f}")

    # write file
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return avg

# ---------- RUN ----------
if not os.path.exists(ANS_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์เฉลย: {ANS_PATH}")
if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์ทำนาย: {PRED_PATH}")

gold = parse_labels_file(ANS_PATH)
pred = parse_labels_file(PRED_PATH)

# Metric A
correct, total = evaluate_metric_A(gold, pred, METRIC_A_PATH)
print(f"[Metric A] First-Word Accuracy: {correct}/{total} = {correct/total:.2%}")
print(f"  -> เขียนผลที่: {METRIC_A_PATH}")

# Metric B
avg_norm = evaluate_metric_B(gold, pred, METRIC_B_PATH)
print(f"[Metric B] Avg normalized edit distance: {avg_norm:.2f}")
print(f"  -> เขียนผลที่: {METRIC_B_PATH}")


[Metric A] First-Word Accuracy: 191/210 = 90.95%
  -> เขียนผลที่: /content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/metric_A2.txt
[Metric B] Avg normalized edit distance: 0.13
  -> เขียนผลที่: /content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/metric_B2.txt


In [ ]:
#save model
model.save('/content/drive/MyDrive/Colab_Notebooks/NLP_Model/ALL_Model/Model_2_V2.h5') #อย่าลืมเปลี่ยน Path ของไฟล์

In [ ]:
#save model
model.save('/content/drive/MyDrive/Colab_Notebooks/NLP_Model/ALL_Model/Model_2_V2.keras') #อย่าลืมเปลี่ยน Path ของไฟล์

# Contest Day

In [12]:
!pip install pythainlp
!pip install gensim
!pip install deepcut

In [13]:
import tensorflow as tf
import numpy as np
# import re
# import io
# import random
# import math
import os
from sklearn.model_selection import train_test_split
from collections import Counter, defaultdict
from pythainlp.tokenize import word_tokenize
from keras.models import load_model
from tensorflow.keras import layers, models

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
CLASS_NAMES = ['E', 'P', 'T']
label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
inverse_label_map = {i: name for i, name in enumerate(CLASS_NAMES)}

In [16]:
def tokenize_sentences(sentences):
  """Tokenizes the sentences using pythainlp."""
  print(sentences)
  tokenized_sentences = {}
  for line_num, sentence in sentences.items():
    tokenized_sentences[line_num] = word_tokenize(sentence , engine = 'deepcut' )
    # tokenized_sentences[line_num] = word_tokenize(sentence  )

  return tokenized_sentences

In [17]:
TARGET_PAT = r'ตา'  # เกณฑ์ง่าย: มี 'ตา' อยู่ที่ไหนในโทเค็นนั้นบ้าง (รวมคำประสม เช่น สายตา, แว่นตา, เบ้าตา ฯลฯ)
WINDOW = 6          # จำนวนโทเค็นซ้าย/ขวาเป็นบริบท
SENTINEL = '[TARGET]'

def find_ta_indices(tokens):
    """คืน index ของทุก token ที่มี 'ตา' เป็นส่วนหนึ่ง."""
    idxs = []
    for i, tok in enumerate(tokens):
        if 'ตา' in tok:  # เกณฑ์ง่าย ตรงกับรูปข้อมูลที่ให้มา
            idxs.append(i)
    return idxs

def build_samples(x_input, y_output):
    texts, labels = [], []
    dropped_mismatch = 0
    per_sent_stats = []

    for line_id, tokens in x_input.items():
        target_positions = find_ta_indices(tokens)
        gold = y_output.get(line_id, [])

        # จัดการจำนวน label ไม่เท่ากับจำนวนตำแหน่ง 'ตา'
        m = min(len(target_positions), len(gold))
        if len(target_positions) != len(gold):
            dropped_mismatch += 1
        # เอาเฉพาะส่วนที่ match ได้
        for k in range(m):
            pos = target_positions[k]
            left = max(0, pos - WINDOW)
            right = min(len(tokens), pos + WINDOW + 1)
            context = tokens[left:pos] + [SENTINEL] + tokens[pos+1:right]
            text = ' '.join(context)  # จะให้ TextVectorization split ด้วย whitespace
            lab = gold[k]
            if lab not in label_map:
                # ข้าม label แปลก
                continue
            texts.append(text)
            labels.append(label_map[lab])

        per_sent_stats.append((line_id, len(target_positions), len(gold), m))

    print(f"จำนวนประโยคทั้งหมด: {len(x_input)}")
    print(f"จำนวนตัวอย่าง (token-level) ที่สร้างได้: {len(texts)}")
    if dropped_mismatch:
        print(f"คำเตือน: มี {dropped_mismatch} ประโยคที่จำนวนตำแหน่ง 'ตา' ไม่เท่ากับจำนวน label (ตัดให้เท่ากันโดยอัตโนมัติ)")

    return texts, np.array(labels, dtype=np.int32)

In [18]:
model =  load_model('/content/drive/MyDrive/Colab_Notebooks/NLP_Model/ALL_Model/Model_1_V2.keras')
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 50)        │          0 │ text[0][0]        │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 50, 128)   │    503,424 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_7         │ (None, 50)        │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │     98,816 │ embedding_3[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 64)        │      8,256 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 3)         │        195 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 610,691 (2.33 MB)

 Trainable params: 610,691 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

## Predict

In [19]:
def predict_labels_for_sentence(tokens):
    """คืน list ของ label ตัวอักษร (E/P/T) ตามจำนวน 'ตา' ที่เจอใน tokens"""
    positions = find_ta_indices(tokens)
    preds = []
    for pos in positions:
        left = max(0, pos - WINDOW)
        right = min(len(tokens), pos + WINDOW + 1)
        context = tokens[left:pos] + [SENTINEL] + tokens[pos+1:right]
        text = ' '.join(context)

        # ⬇️ แก้บรรทัดนี้
        # prob = model.predict([text], verbose=0)[0]
        prob = model.predict(np.array([text], dtype=object), verbose=0)[0]
        # หรือใช้อีกแบบ:
        # prob = model(np.array([text], dtype=object)).numpy()[0]

        pred_idx = int(np.argmax(prob))
        preds.append(inverse_label_map[pred_idx])
    return preds



def write_predictions(input_file, output_file):
    """อ่านประโยคจาก input_file (รูปแบบ id::ประโยค) แล้วเขียนผลพยากรณ์ลง output_file"""
    # 1) อ่านเป็น dict: {id(int): sentence(str)}
    sentences_only = {}
    with open(input_file, 'r', encoding='utf-8') as fin:
        for line in fin:
            line = line.strip()
            if not line or "::" not in line:
                continue
            line_id, text = line.split("::", 1)
            sentences_only[int(line_id)] = text.strip()

    # 2) ส่ง 'dict' เข้า tokenizer (ฟังก์ชันของคุณต้องการ .items())
    #    ผลลัพธ์คาดว่าเป็น {id: [tokens...]}
    tokenized_all = tokenize_sentences(sentences_only)
    print("Tokenized sentences:", tokenized_all)

    # 3) ทำนายและเขียนผลเป็น id::label[,label...]
    with open(output_file, 'w', encoding='utf-8') as fout:
        for line_id in sorted(tokenized_all.keys()):
            tokens = tokenized_all[line_id]
            preds = predict_labels_for_sentence(tokens)
            out = f"{line_id}::" + (",".join(preds) if preds else "")
            fout.write(out + "\n")

    print(f"เขียนผลลัพธ์แล้วที่: {os.path.abspath(output_file)}")

ส่วนของการ Predict จาก model ที่ Train มา

In [22]:

# ใส้  input file ตรงนี้
src_file = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/input.txt'      # ไฟล์ input
pred_file = '/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/ans.txt'      # ผลลัพธ์ต้องการให้ชื่อ pred.txt ตามที่ระบุ

write_predictions(src_file, pred_file)

# แสดงผล 20 บรรทัดแรกของ pred.txt
print("\n=== ตัวอย่างผลลัพธ์จาก pred.txt ===")
with open(pred_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(line.strip())


{1: 'กระจกตาของเขาสีฟ้าข้างหนึ่ง สีเขียวข้างหนึ่ง', 2: 'การเดินหมากตานี้ทำให้ผมมองโกะในมุมใหม่', 3: 'การที่ลูกได้รดน้ำดำหัวปู่ย่าตายาย จะส่งเสริมพัฒนาการด้านใดบ้าง', 4: 'การปิดตาตีหม้อนิยมเล่นกันมานานโดยเฉพาะในเทศกาลวันสงกรานต์เพราะเป็นการละเล่นที่สนุกทั้งคนเล่นและคนดู ฝึกความจำ การสังเกตทิศทางและการกะระยะทาง', 5: 'การผ่าตัดเปลี่ยนกระจกตาสำเร็จเป็นครั้งแรกนั้นไม่ใช่ตาของคนเรา แต่เป็นตาของสัตว์', 6: 'การเล่นหมากตานี้เป็นไปอย่างดุเดือด เบี้ยฝ่ายขาวโจมตีทหารราบฝ่ายดำและเข้ายึดพื้นที่ ฝ่ายดำไม่รอช้าสวนทหารม้าเข้ามาสกัดทัพ', 7: "เก่ง การุณ' โพสต์มีคนตาดีเห็น 'ลุงข้างบ้าน' ตีกอล์ฟที่อยุธยา", 8: 'เกมนี้คนที่ตาไม่ดีอย่างเธอเล่นกี่ตาก็แพ้', 9: 'ใกล้จะปีใหม่เเล้วซื้อของขวัญอะไรให้คุณตาคุณยายดีคะ', 10: 'ขอตาหน้าอีกตาเดียว กูเห็นมึงพูดแบบนี้มาทุกตา 555+', 11: 'ขอวิธีรักษาอาการตาแห้งหน่อยคับ คุณตาของผมบ่นตลอดว่าเวลาโดนแอร์ ตาแกจะเคืองมาก ๆ เลยคับ', 12: 'ขออัญเชิญปีศาจสามตาลงบนสนาม เป็นอันจบตานี้', 13: 'ข้าขอล้างแค้นให้กับท่านตาของข้า ที่แพ้หมากล้อมให้กับเจ้าในตานั้น', 14: 'ขี้หลงขี้ลืมเหมือนตาแก่',

## ประเมินผล Confition Marix

In [ ]:
def parse_labels_file(path):
    """
    อ่านไฟล์รูปแบบ:
        1::E
        2::P
        3::T,T
    แล้วคืนเป็น dict: {int_id: [label, label, ...]}
    ถ้าหลัง '::' ว่าง ให้เป็นลิสต์ว่าง []
    """
    data = {}
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or "::" not in line:
                continue
            k, v = line.split("::", 1)
            k = int(k.strip())
            v = v.strip()
            if not v:
                labels = []
            else:
                labels = [lab.strip() for lab in v.split(",") if lab.strip()]
            data[k] = labels
    return data

def evaluate_files(gold_dict, pred_dict, label_set=VALID_LABELS):
    """
    ประเมิน 2 แบบ:
    (A) Micro accuracy ต่อ occurrence: เทียบ element-wise โดยใช้จำนวน occurrence ของ gold เป็นฐาน
        - ถ้า pred เกิน: ส่วนเกินนับเป็นผิด
        - ถ้า pred ขาด: ส่วนที่ขาดนับเป็นผิด
    (B) Exact-match accuracy ต่อบรรทัด: บรรทัดถูกเมื่อ sequence ของ label เหมือนกันเป๊ะ

    เพิ่มเติม: สรุป confusion ต่อ label (เฉพาะตำแหน่งที่มี gold)
    """
    ids = sorted(set(gold_dict.keys()) | set(pred_dict.keys()))
    total_occ = 0
    correct_occ = 0
    exact_match = 0

    # confusion: gold->pred นับเฉพาะตำแหน่งที่มี gold label
    conf = Counter()

    # เก็บสถิติความยาวไม่เท่ากัน
    length_mismatch = 0

    for i in ids:
        gold = gold_dict.get(i, [])
        pred = pred_dict.get(i, [])

        # นับ exact-match ต่อบรรทัด
        if gold == pred:
            exact_match += 1
        else:
            if len(gold) != len(pred):
                length_mismatch += 1

        # นับ micro ต่อ occurrence (ใช้ gold เป็นฐาน)
        Lg = len(gold)
        Lp = len(pred)

        # คู่ที่เทียบกันได้ (min)
        Lc = min(Lg, Lp)
        for j in range(Lc):
            g = gold[j]
            p = pred[j]
            # ทำความสะอาด label ที่นอกชุด (กันพลาด)
            if g not in label_set:
                # ถ้า gold นอกชุด ก็ยังนับเป็นหนึ่ง occurrence แต่จะไม่ลง confusion เพื่อความปลอดภัย
                total_occ += 1
                correct_occ += int(g == p)
                continue
            total_occ += 1
            if g == p:
                correct_occ += 1
            conf[(g, p)] += 1

        # ส่วนเกิน/ขาด คิดเป็นผิดทั้งหมด
        if Lp > Lg:
            # pred เกิน
            total_occ += (Lp - Lg)
            # นับผิด ไม่ต้องลง confusion เพราะไม่มี gold ให้จับคู่
        elif Lg > Lp:
            # pred ขาด
            total_occ += (Lg - Lp)
            # นับผิดเช่นกัน

    micro_acc = (correct_occ / total_occ) if total_occ else 0.0
    line_acc = (exact_match / len(ids)) if ids else 0.0

    # ทำ confusion matrix ขนาด |labels| x |labels|
    labels_sorted = sorted(label_set)
    conf_mat = np.zeros((len(labels_sorted), len(labels_sorted)), dtype=int)
    idx = {lab: k for k, lab in enumerate(labels_sorted)}
    for (g, p), c in conf.items():
        if g in idx and p in idx:
            conf_mat[idx[g], idx[p]] += c

    report = {
        "num_lines": len(ids),
        "exact_match_lines": exact_match,
        "length_mismatch_lines": length_mismatch,
        "total_occurrences_gold+penalty": total_occ,
        "correct_occurrences": correct_occ,
        "micro_accuracy": micro_acc,
        "line_exact_match_accuracy": line_acc,
        "labels_order": labels_sorted,
        "confusion_matrix": conf_mat,
    }
    return report


In [ ]:
# ===== Cell: Evaluate predictions against gold files (ansTest.txt vs pred.txt) =====


ANS_PATH  = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_ans.txt"      # ไฟล์เฉลยจริง
PRED_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/valid_pred.txt"     # ไฟล์ทำนายจากโมเดล
VALID_LABELS = {"E", "P", "T"}  # ชุด label ที่ยอมรับ

# โหลดไฟล์
if not os.path.exists(ANS_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์เฉลย: {ANS_PATH}")
if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์พยากรณ์: {PRED_PATH}")

gold = parse_labels_file(ANS_PATH)
pred = parse_labels_file(PRED_PATH)

report = evaluate_files(gold, pred, label_set=VALID_LABELS)

# แสดงผล
print("=== Evaluation Report ===")
print(f"- #Lines: {report['num_lines']}")
print(f"- Exact-match lines: {report['exact_match_lines']} / {report['num_lines']} "
      f"({report['line_exact_match_accuracy']*100:.2f}%)")
print(f"- Length-mismatch lines: {report['length_mismatch_lines']}")
print(f"- Occurrences (gold-based + penalties): {report['total_occurrences_gold+penalty']}")
print(f"- Correct occurrences: {report['correct_occurrences']} "
      f"({report['micro_accuracy']*100:.2f}%)")

# Confusion matrix
print("\nConfusion Matrix (rows=Gold, cols=Pred):")
labs = report["labels_order"]
mat = report["confusion_matrix"]
header = "      " + "  ".join(f"{l:>4s}" for l in labs)
print(header)
for i, g in enumerate(labs):
    row = "  ".join(f"{mat[i, j]:>4d}" for j in range(len(labs)))
    print(f"{g:>4s}  {row}")

# ความแม่นยำราย label (เฉพาะตำแหน่งที่มี gold=label นั้น)
per_label = {}
for li, g in enumerate(labs):
    support = mat[li, :].sum()
    correct = mat[li, li]
    acc_g = (correct / support) if support else 0.0
    per_label[g] = (correct, support, acc_g)

print("\nPer-Label Accuracy (conditioned on gold):")
for g in labs:
    correct, support, acc_g = per_label[g]
    print(f"- {g}: {correct}/{support} = {acc_g*100:.2f}%")


=== Evaluation Report ===
- #Lines: 210
- Exact-match lines: 168 / 210 (80.00%)
- Length-mismatch lines: 14
- Occurrences (gold-based + penalties): 341
- Correct occurrences: 294 (86.22%)

Confusion Matrix (rows=Gold, cols=Pred):
         E     P     T
   E   136    12     1
   P    12    74     5
   T     0     3    84

Per-Label Accuracy (conditioned on gold):
- E: 136/149 = 91.28%
- P: 74/91 = 81.32%
- T: 84/87 = 96.55%


## การประเมินผลแบบ Matrix A , B

In [24]:
# ---------- PATHS ----------
ANS_PATH   = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/ans.txt" # คำตอบของ input
PRED_PATH  = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/ans_pred.txt" # pred
METRIC_A_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/metric_A2.txt"
METRIC_B_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/metric_B2.txt"

VALID_LABELS = {"E", "P", "T"}

# ---------- IO HELPERS ----------
def parse_labels_file(path):
    """
    อ่านไฟล์รูปแบบ:
        1::E,E
        2::P
        3::T,T
    แล้วคืน dict: {int_id: [label, ...]}
    """
    data = {}
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or "::" not in line:
                continue
            k, v = line.split("::", 1)
            k = int(k.strip())
            v = v.strip()
            if not v:
                labels = []
            else:
                labels = [lab.strip() for lab in v.split(",") if lab.strip()]
            data[k] = labels
    return data

# ---------- METRIC A ----------
def evaluate_metric_A(gold_dict, pred_dict, out_path):
    """
    First-Word Accuracy:
      - ต่อบรรทัด: "id:: Correct" ถ้า label ตัวแรกตรงกันและมีอยู่ทั้งคู่ มิฉะนั้น "Wrong"
      - สุดท้าย: "Accuracy::X/Y"
    เกณฑ์:
      - ถ้าฝั่งใดฝั่งหนึ่งไม่มี label ตัวแรก (ลิสต์ว่าง) => Wrong
    """
    ids = sorted(set(gold_dict.keys()) | set(pred_dict.keys()))
    correct = 0
    total = len(ids)

    lines = []
    for i in ids:
        gold = gold_dict.get(i, [])
        pred = pred_dict.get(i, [])
        if gold and pred and gold[0] == pred[0]:
            lines.append(f"{i}:: Correct")
            correct += 1
        else:
            lines.append(f"{i}:: Wrong")

    lines.append(f"Accuracy::{correct}/{total}")

    # write file
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return correct, total

# ---------- Levenshtein (token-level) ----------
def levenshtein_ops(seq_a, seq_b):
    """
    Levenshtein distance สำหรับลิสต์ของสัญลักษณ์ (เช่น ['E','E'] กับ ['P','E'])
    คืนจำนวน operations (insert/delete/replace) ขั้นต่ำเพื่อแปลง seq_a -> seq_b
    """
    na, nb = len(seq_a), len(seq_b)
    dp = [[0]*(nb+1) for _ in range(na+1)]
    for i in range(na+1):
        dp[i][0] = i
    for j in range(nb+1):
        dp[0][j] = j
    for i in range(1, na+1):
        for j in range(1, nb+1):
            cost = 0 if seq_a[i-1] == seq_b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # delete
                dp[i][j-1] + 1,      # insert
                dp[i-1][j-1] + cost  # replace / match
            )
    return dp[na][nb]

# ---------- METRIC B ----------
def evaluate_metric_B(gold_dict, pred_dict, out_path):
    """
    Metric B: Edit distance (token-level) ต่อบรรทัด + ค่าเฉลี่ย normalized
      - รายบรรทัด: "id:: <ops>/<len_pred>"
          * ops = Levenshtein distance(GOLD, PRED)
          * len_pred = จำนวน token ใน pred บรรทัดนั้น (รายงานตามจริง; ถ้า 0 จะพิมพ์ 0/0)
      - Avg::X.YZ (เฉลี่ย ops / max(1, len_pred))
        * ใช้ max(1, len_pred) เพื่อเลี่ยงหารศูนย์ แต่ในไฟล์บรรทัดยังคงโชว์ตัวหารจริง
    """
    ids = sorted(set(gold_dict.keys()) | set(pred_dict.keys()))
    norm_sum = 0.0
    count = 0

    lines = []
    for i in ids:
        gold = gold_dict.get(i, [])
        pred = pred_dict.get(i, [])
        ops = levenshtein_ops(gold, pred)
        denom_real = len(pred)               # แสดงผลตามที่ผู้ใช้ต้องการ
        denom_for_avg = max(1, denom_real)   # ป้องกันหารศูนย์ตอนเฉลี่ย

        lines.append(f"{i}:: {ops}/{denom_real}")
        norm_sum += ops / denom_for_avg
        count += 1

    avg = (norm_sum / count) if count else 0.0
    lines.append(f"Avg::{avg:.2f}")

    # write file
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return avg

# ---------- RUN ----------
if not os.path.exists(ANS_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์เฉลย: {ANS_PATH}")
if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(f"ไม่พบไฟล์ทำนาย: {PRED_PATH}")

gold = parse_labels_file(ANS_PATH)
pred = parse_labels_file(PRED_PATH)

# Metric A
correct, total = evaluate_metric_A(gold, pred, METRIC_A_PATH)
print(f"[Metric A] First-Word Accuracy: {correct}/{total} = {correct/total:.2%}")
print(f"  -> เขียนผลที่: {METRIC_A_PATH}")

# Metric B
avg_norm = evaluate_metric_B(gold, pred, METRIC_B_PATH)
print(f"[Metric B] Avg normalized edit distance: {avg_norm:.2f}")
print(f"  -> เขียนผลที่: {METRIC_B_PATH}")


[Metric A] First-Word Accuracy: 156/200 = 78.00%
  -> เขียนผลที่: /content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/metric_A2.txt
[Metric B] Avg normalized edit distance: 0.24
  -> เขียนผลที่: /content/drive/MyDrive/Colab_Notebooks/NLP_Model/DataSet_Testing/Contest_Day/metric_B2.txt
